# Brand YOLO Pipeline Run

Prepare brand data and train the YOLO brand detector.

Steps:
- Prepare YOLO dataset from brand data.
- Train the YOLO brand detector.
- Verify output artifacts.



In [ ]:
from __future__ import annotations

import json
import os
import sys
import subprocess
from pathlib import Path

# Resolve repo root from the notebook location.
REPO_ROOT = Path.cwd()
for parent in [REPO_ROOT] + list(REPO_ROOT.parents):
    if (parent / 'scripts').exists() and (parent / 'notebooks').exists():
        REPO_ROOT = parent
        break

# Ensure local modules are importable.
sys.path.insert(0, str(REPO_ROOT))
sys.path.insert(0, str(REPO_ROOT / 'src'))

PY = sys.executable

def run(cmd: list[str]) -> None:
    # Run a command from the repo root with PYTHONPATH set.
    env = os.environ.copy()
    env['PYTHONPATH'] = os.pathsep.join([str(REPO_ROOT / 'src'), str(REPO_ROOT)])
    print('$', ' '.join(cmd))
    subprocess.run(cmd, cwd=str(REPO_ROOT), check=True, env=env)

def show_json(rel_path: str) -> None:
    path = REPO_ROOT / rel_path
    if not path.exists():
        print('Missing:', path)
        return
    try:
        data = json.loads(path.read_text(encoding='utf-8'))
    except Exception:
        print(path.read_text(encoding='utf-8', errors='ignore')[:2000])
        return
    print(json.dumps(data, indent=2))

def list_dir(rel_path: str, limit: int = 20) -> None:
    path = REPO_ROOT / rel_path
    if not path.exists():
        print('Missing:', path)
        return
    print(f'\n{rel_path}/')
    for item in sorted(path.iterdir())[:limit]:
        print(' -', item.name)


In [ ]:
summary = {
    'dataset': {},
    'artifact': None,
}

run([PY, 'scripts/prepare_brand_data.py'])
run([PY, '-m', 'src.train.train_brand_logo_detector'])


In [ ]:
# Inspect prepared dataset counts.
brand_root = REPO_ROOT / 'data' / 'processed' / 'brand_yolo'
if brand_root.exists():
    for split in ['train', 'val']:
        img_dir = brand_root / 'images' / split
        lbl_dir = brand_root / 'labels' / split
        img_count = len([p for p in img_dir.rglob('*') if p.is_file()]) if img_dir.exists() else 0
        lbl_count = len([p for p in lbl_dir.rglob('*') if p.is_file()]) if lbl_dir.exists() else 0
        summary['dataset'][split] = {'images': img_count, 'labels': lbl_count}
        print(split, summary['dataset'][split])
else:
    print('Missing:', brand_root)


In [ ]:
# Verify trained artifact.
artifact = REPO_ROOT / 'artifacts' / 'brand' / 'yolo_logo_det.pt'
if artifact.exists():
    summary['artifact'] = {
        'path': str(artifact.relative_to(REPO_ROOT)),
        'size_mb': round(artifact.stat().st_size / 1024**2, 2),
    }
    print('Artifact:', summary['artifact'])
else:
    print('Missing:', artifact)


In [ ]:
# Persist summary for quick reference.
report_dir = REPO_ROOT / 'reports'
report_dir.mkdir(parents=True, exist_ok=True)
summary_path = report_dir / 'execution_brand_yolo_summary.json'
summary_path.write_text(json.dumps(summary, indent=2))
print('Saved summary to', summary_path)


In [ ]:
# Quick artifact index for verification.
for folder in ['models', 'experiments', 'artifacts', 'runs', 'reports', 'logs']:
    path = REPO_ROOT / folder
    if not path.exists():
        continue
    print(f'\n{folder}/')
    for item in sorted(path.iterdir())[:20]:
        print(' -', item.name)
